# 第 1 周末练习 —— 技术问答解释器（解答）

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请构建一个工具：

- **输入**：一个技术问题（例如某段 Python 在干什么）
- **输出**：清晰解释
- **双后端**：同一问题分别走 **OpenRouter**（云端 `gpt-4o-mini`）与 **Ollama**（本地 `llama3.2`）

这是你在课程期间可以亲自使用的工具！

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| OpenAI 兼容 `base_url` | OpenRouter 与 Ollama 都复用 `OpenAI` 客户端 |
| 云端模型 | `MODEL_GPT = 'gpt-4o-mini'`（经 OpenRouter） |
| 本地开源模型 | `MODEL_LLAMA = 'llama3.2'`（经 Ollama `/v1`） |
| Markdown 展示 | `display(Markdown(result))` |

## 怎么跑

1. `.env` 准备 `OPENROUTER_API_KEY`；本机启动 Ollama 并确保能访问 `http://localhost:11434`
2. 从上到下运行；配置格会 `ollama pull llama3.2`
3. 改 `question` 后分别跑 GPT / Llama 两格做对比


In [1]:
# ========== 导入：后面双后端问答要用的工具箱 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENROUTER_API_KEY
import os
# 导入 requests：用 HTTP 探活本机 Ollama 根路径是否在线
import requests
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# Markdown / display / update_display：在 Jupyter 里展示（本解答主要用 display+Markdown）
from IPython.display import Markdown, display, update_display
# OpenAI 客户端：这里同时给 OpenRouter 与 Ollama 的兼容端点用
from openai import OpenAI


In [2]:
# ========== 常量：模型名集中定义，后面只改这里 ==========

# 常量

# OpenRouter / 云端侧使用的模型 id（字符串必须与路由侧支持的名字一致）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 pull；须与本机已安装名字一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境配置：OpenRouter + Ollama 两个 OpenAI 兼容客户端 ==========

# 配置环境
# override=True：以 .env 文件为准覆盖已有同名环境变量
load_dotenv(override=True)

# 从环境读取 OpenRouter API Key
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
# 有密钥就打印前 3 个字符做存在性确认（不要打印完整密钥）
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    # 未设置时的提示文案保持英文原样
    print("OpenRouter API Key not set")

# OpenRouter 的 OpenAI 兼容 API 根地址
openrouter_url = "https://openrouter.ai/api/v1"
# 本地 Ollama 的 OpenAI 兼容端点（注意是 /v1，不是原生 /api/chat）
ollama_url = "http://localhost:11434/v1"


# 云端客户端：base_url 指 OpenRouter，密钥用 OPENROUTER_API_KEY
openai = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

# 本地客户端：Ollama 兼容模式常把 api_key 写成占位 "ollama"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# 探活：GET 本机 Ollama 根路径，看服务是否起来（.content 为响应字节）
requests.get("http://localhost:11434/").content

# IPython shell magic：拉取本地 llama3.2 模型（需本机已装 ollama CLI）
!ollama pull llama3.2


In [ ]:
# ========== 提问：改三引号里的问题即可换题 ==========

# 第 1 周解答：逻辑与字符串保持原样
# 在此填写问题；覆盖即可提问新问题

# 发给模型的 user 内容保持英文（影响回答的字符串不翻译）
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 用 gpt-4o-mini（经 OpenRouter）回答并 Markdown 展示 ==========

# 用 gpt-4o-mini 流式回答（注：本格实际未传 stream=True，是一次性非流式调用；逻辑保持原样）
response = openai.chat.completions.create(
  model=MODEL_GPT,
  messages=[
            # 仅 user：把整段 question 当作用户消息
            {"role": "user", "content": question},
        ])
# 取出第一条候选的文本
result = response.choices[0].message.content
# 在笔记本里用 Markdown 渲染完整回答
display(Markdown(result))


In [ ]:
# ========== 用本地 Llama 3.2（经 Ollama 兼容端点）回答 ==========

# 用 Llama 3.2 回答
# 同一个 Chat Completions 形状，只是客户端换成 ollama、model 换成 MODEL_LLAMA
response = ollama.chat.completions.create(
  model=MODEL_LLAMA,
  messages=[{"role": "user", "content": question}]
)

# 取出文本并用 Markdown 展示，便于和上一格云端回答对比
result = response.choices[0].message.content
display(Markdown(result))
